List of stocks and returns historical construct an optimal portfolio.
Given that the return (Assume a return) advice what is the lowest risk portfolio.
Mean variance portfolio
Normally distributed portfolio
Given the risk tolerance what is the best return ETFs


Markowitz package


In [2]:
!pip install empyrical

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for empyrical: filename=empyrical-0.5.5-py3-none-any.whl size=39753 sha256=5dd06a2f7b02805b8f3583aef968d5d3e255b6eee9428de7170c92950ccb2df4
  Stored in directory: /root/.cache/pip/wheels/ac/1d/58/a7ae5ef5c8de7c4b769f24c2584f4706564921f031b16b9cb6
Successfully built empyrical


#FINAL CODE

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from empyrical import max_drawdown

# Load ETF Data
def load_etf_data(files):
    data = {}
    for etf, file in files.items():
        df = pd.read_csv(file, parse_dates=['Date'])
        df.set_index('Date', inplace=True)
        df.rename(columns={'Return': etf}, inplace=True)
        df[etf] = df[etf] / 100  # Convert percentage returns to decimal
        data[etf] = df[etf]
    return pd.DataFrame(data).dropna()

# Compute portfolio statistics
def portfolio_statistics(weights, mean_returns, cov_matrix):
    portfolio_return = np.dot(weights, mean_returns)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)  # Annualized volatility
    return portfolio_return, portfolio_volatility

# Objective function for maximizing return under risk constraint
def max_return(weights, mean_returns, cov_matrix, risk_tolerance):
    portfolio_return, portfolio_volatility = portfolio_statistics(weights, mean_returns, cov_matrix)
    if portfolio_volatility > risk_tolerance:
        return np.inf  # Penalize solutions that exceed the risk tolerance
    return -portfolio_return  # Negative sign since we maximize returns

# Find optimal portfolio with a risk tolerance constraint for each year
def optimal_portfolio(returns_df, risk_tolerance):
    years = sorted(returns_df.index.year.unique())  # Get unique years in dataset
    yearly_weights = {}
    yearly_returns = {}

    for year in years:
        year_data = returns_df.loc[str(year)]
        if year_data.empty:
            continue

        mean_returns = year_data.mean()
        cov_matrix = year_data.cov()
        num_assets = len(mean_returns)

        initial_weights = np.ones(num_assets) / num_assets
        bounds = tuple((0, 1) for _ in range(num_assets))  # Allow weights to be 0
        constraints = [
            {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},  # Sum of weights = 1
        ]

        result = minimize(max_return, initial_weights, args=(mean_returns, cov_matrix, risk_tolerance),
                          method='SLSQP', bounds=bounds, constraints=constraints)

        if not result.success:
            # If risk tolerance is too restrictive, fall back to minimizing volatility
            result = minimize(lambda w: portfolio_statistics(w, mean_returns, cov_matrix)[1],
                              initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)

        weights = np.array(result.x) / np.sum(result.x)  # Normalize weights
        yearly_weights[year] = weights

        # Compute returns using the optimized weights
        yearly_returns[year] = year_data.dot(weights)

        print(f"Optimal Portfolio Weights for {year} (Risk Tolerance: {risk_tolerance}):")
        print(pd.DataFrame({'ETF': returns_df.columns, 'Weight': weights}))
        print("-" * 50)

    return yearly_weights, yearly_returns

# Compute annual statistics
def compute_annual_statistics(df, yearly_returns, initial_cash=100):
    annual_stats = []
    cash = initial_cash

    for year, strategy_returns in yearly_returns.items():
        if strategy_returns.empty:
            continue

        annual_return = (1 + strategy_returns).prod() - 1  # Compounded annual return
        cash *= (1 + annual_return)  # Convert to cash value
        annual_volatility = strategy_returns.std() * np.sqrt(252)
        sharpe_ratio = annual_return / annual_volatility if annual_volatility > 0 else np.nan
        mdd = max_drawdown(strategy_returns)

        annual_stats.append([year, cash, annual_return, annual_volatility, sharpe_ratio, mdd])

    return pd.DataFrame(annual_stats, columns=['Year', 'Cash Value', 'Annual Return', 'Volatility', 'Sharpe Ratio', 'Max Drawdown'])

# Load Data
etf_files = {
    'xlb': "XLB_weekly_return_detailed.csv",
    'xle': "XLE_weekly_return_detailed.csv",
    'xlf': "XLF_weekly_return_detailed.csv",
    'xli': "XLI_weekly_return_detailed.csv",
    'xlk': "XLK_weekly_return_detailed.csv",
    'xlp': "XLP_weekly_return_detailed.csv",
    'xlu': "XLU_weekly_return_detailed.csv",
    'xlv': "XLV_weekly_return_detailed.csv",
    'xly': "XLY_weekly_return_detailed.csv"
}

daily_returns = load_etf_data(etf_files)

# User-specified risk tolerance
risk_tolerance = float(input("Enter your risk tolerance (e.g., 0.15 for 15%): "))

# Compute Markowitz Strategy Returns **Yearly** with Risk Tolerance
yearly_weights, yearly_returns = optimal_portfolio(daily_returns, risk_tolerance)

# Store yearly weights
weights_df = pd.DataFrame.from_dict(yearly_weights, orient='index', columns=daily_returns.columns)
weights_df.index.name = 'Year'
weights_df.to_csv("markowitz_yearly_weights.csv")

# Compute Annual Statistics in Cash
markowitz_stats = compute_annual_statistics(daily_returns, yearly_returns)

# Save Results to CSV
markowitz_stats.to_csv("markowitz_annual_cash.csv", index=False)

# Display the final comparison DataFrame
print("Markowitz Annual Returns in Cash:")
print(markowitz_stats)


Enter your risk tolerance (e.g., 0.15 for 15%): 0.15


/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df

Optimal Portfolio Weights for 1999 (Risk Tolerance: 0.15):
   ETF        Weight
0  xlb  1.070583e-01
1  xle  1.528923e-01
2  xlf  0.000000e+00
3  xli  2.183355e-01
4  xlk  0.000000e+00
5  xlp  1.900472e-01
6  xlu  2.276283e-01
7  xlv  1.040385e-01
8  xly  2.298509e-17
--------------------------------------------------
Optimal Portfolio Weights for 2000 (Risk Tolerance: 0.15):
   ETF        Weight
0  xlb  5.092981e-02
1  xle  1.836215e-01
2  xlf  1.043545e-17
3  xli  5.746272e-18
4  xlk  8.826391e-02
5  xlp  2.483410e-01
6  xlu  2.564013e-01
7  xlv  1.724425e-01
8  xly  0.000000e+00
--------------------------------------------------
Optimal Portfolio Weights for 2001 (Risk Tolerance: 0.15):
   ETF        Weight
0  xlb  2.581754e-02
1  xle  8.179329e-02
2  xlf  1.883915e-02
3  xli  8.971773e-18
4  xlk  0.000000e+00
5  xlp  3.574965e-01
6  xlu  3.438442e-01
7  xlv  1.542199e-01
8  xly  1.798943e-02
--------------------------------------------------
Optimal Portfolio Weights for 2002 (Risk

/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0


Optimal Portfolio Weights for 2009 (Risk Tolerance: 0.15):
   ETF        Weight
0  xlb  3.266326e-17
1  xle  0.000000e+00
2  xlf  1.539546e-17
3  xli  2.491327e-18
4  xlk  0.000000e+00
5  xlp  5.998677e-01
6  xlu  2.125955e-01
7  xlv  1.875368e-01
8  xly  0.000000e+00
--------------------------------------------------
Optimal Portfolio Weights for 2010 (Risk Tolerance: 0.15):
   ETF        Weight
0  xlb  1.088185e-16
1  xle  7.319911e-19
2  xlf  3.592345e-17
3  xli  1.073402e-16
4  xlk  0.000000e+00
5  xlp  9.590551e-01
6  xlu  4.094491e-02
7  xlv  0.000000e+00
8  xly  2.279523e-17
--------------------------------------------------
Optimal Portfolio Weights for 2011 (Risk Tolerance: 0.15):
   ETF        Weight
0  xlb  0.000000e+00
1  xle  4.702889e-17
2  xlf  0.000000e+00
3  xli  0.000000e+00
4  xlk  1.891383e-16
5  xlp  8.755635e-01
6  xlu  1.244365e-01
7  xlv  0.000000e+00
8  xly  1.954565e-16
--------------------------------------------------
Optimal Portfolio Weights for 2012 (Risk

/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_numdiff.py:590: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0


In [9]:
print("Markowitz Annual Returns in Cash:")
print(markowitz_stats.describe())  # Summary statistics

Markowitz Annual Returns in Cash:
              Year   Cash Value  Annual Return  Volatility  Sharpe Ratio  \
count    26.000000    26.000000      26.000000   26.000000     26.000000   
mean   2011.500000   503.713364       0.132069    0.141402      1.115390   
std       7.648529   521.028141       0.135921    0.043623      1.054417   
min    1999.000000    89.874187      -0.184684    0.062760     -0.967602   
25%    2005.250000   150.965708       0.096654    0.116743      0.461909   
50%    2011.500000   258.084486       0.147219    0.137296      1.154285   
75%    2017.750000   634.136273       0.206853    0.149993      1.692189   
max    2024.000000  2071.154641       0.355840    0.277211      3.062951   

       Max Drawdown  
count     26.000000  
mean      -0.113688  
std        0.065278  
min       -0.280115  
25%       -0.128597  
50%       -0.098585  
75%       -0.065508  
max       -0.025664  
